# DoorDash Delivery ETA — Etapa 03: Análise Exploratória de Dados (EDA)

Este notebook investiga os fatores associados à variável-alvo `delivery_duration_minutes` utilizando os `197.421` registros com alvo válido gerados pelo pipeline controlado em `src/data_preparation.py` (`df_model`).

> **Regras desta etapa:** Não há imputação, clipping, remoção de outliers ou treinamento de modelos. Todas as análises informam o tamanho da amostra (`n`) e distinguem resultados observados de hipóteses.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 35)
pd.set_option("display.width", 140)

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_preparation import prepare_datasets

RAW_PATH = PROJECT_ROOT / "data" / "raw" / "historical_data.csv"
FIG_DIR = PROJECT_ROOT / "reports" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

_, _, df, summary = prepare_datasets(RAW_PATH)
print(f"Registros válidos analisados (df_model): {len(df)} | Colunas: {df.shape[1]}")

## 1. Distribuição da Variável-Alvo (`delivery_duration_minutes`)

In [ ]:
target = df["delivery_duration_minutes"]
pcts = [0.50, 0.75, 0.90, 0.95, 0.99, 0.999]
display(pd.DataFrame({
    "percentil": [f"P{p*100:g}" for p in pcts],
    "minutos": [round(target.quantile(p), 2) for p in pcts]
}))

bins = [0, 30, 45, 60, 90, 120, np.inf]
labels = ["<= 30 min", "30-45 min", "45-60 min", "60-90 min", "90-120 min", "> 120 min"]
df["duration_band"] = pd.cut(target, bins=bins, labels=labels, right=True)
band_tbl = df["duration_band"].value_counts().reindex(labels).to_frame("n")
band_tbl["%"] = (band_tbl["n"] / len(df) * 100).round(2)
display(band_tbl)

# Inspeção dos extremos (> 24h) sem removê-los
display(df[df["is_duration_gt_24h"]][["created_at", "actual_delivery_time", "delivery_duration_minutes", "market_id", "store_id", "store_primary_category"]])

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
sns.histplot(df.loc[target <= 150, "delivery_duration_minutes"], bins=60, kde=True, ax=axes[0, 0], color="#2b5c8f")
axes[0, 0].set_title("Distribuição da Duração (<= 150 min)")
axes[0, 0].set_xlabel("Duração (minutos)")

sns.histplot(np.log10(target), bins=60, kde=True, ax=axes[0, 1], color="#1f77b4")
axes[0, 1].set_title("Distribuição em Escala Log10 (minutos)")
axes[0, 1].set_xlabel("log10(Duração em minutos)")

sns.boxplot(y=df.loc[target <= 180, "delivery_duration_minutes"], ax=axes[1, 0], color="#55a868")
axes[1, 0].set_title("Boxplot da Duração (<= 180 min)")
axes[1, 0].set_ylabel("Duração (minutos)")

sns.ecdfplot(df.loc[target <= 150, "delivery_duration_minutes"], ax=axes[1, 1], color="#c44e52")
axes[1, 1].set_title("ECDF da Duração (<= 150 min)")
axes[1, 1].set_xlabel("Duração (minutos)")
fig.tight_layout()
plt.show()

## 2. Análise Temporal

In [ ]:
df["created_day"] = df["created_at"].dt.day
def get_period(h):
    if 0 <= h <= 5: return "Madrugada (00-05h)"
    if 6 <= h <= 11: return "Manha (06-11h)"
    if 12 <= h <= 17: return "Tarde (12-17h)"
    return "Noite (18-23h)"
df["period_of_day"] = df["created_hour"].apply(get_period)

def agg_stats(col):
    g = df.groupby(col, observed=False)["delivery_duration_minutes"]
    return pd.DataFrame({
        "n": g.count(),
        "mediana": g.median().round(2),
        "media": g.mean().round(2),
        "std": g.std().round(2),
        "iqr": (g.quantile(0.75) - g.quantile(0.25)).round(2),
    })

display(agg_stats("created_hour"))
display(agg_stats("created_dayofweek"))
display(agg_stats("period_of_day"))
display(agg_stats("created_is_weekend"))
display(agg_stats("created_day").sort_values("mediana", ascending=False).head(10))

## 3. Características dos Pedidos

In [ ]:
order_cols = [
    "total_items", "num_distinct_items", "subtotal", "min_item_price",
    "max_item_price", "estimated_order_place_duration", "estimated_store_to_consumer_driving_duration"
]
corr_rows = []
for col in order_cols:
    sub = df[[col, "delivery_duration_minutes"]].dropna()
    corr_rows.append({
        "variavel": col, "n": len(sub),
        "pearson": round(sub[col].corr(sub["delivery_duration_minutes"], method="pearson"), 4),
        "spearman": round(sub[col].corr(sub["delivery_duration_minutes"], method="spearman"), 4),
    })
display(pd.DataFrame(corr_rows))

df["subtotal_q"] = pd.qcut(df["subtotal"], q=4, duplicates="drop")
df["driving_q"] = pd.qcut(df["estimated_store_to_consumer_driving_duration"], q=4, duplicates="drop")
display(agg_stats("subtotal_q"))
display(agg_stats("driving_q"))

## 4. Análise Operacional

In [ ]:
op_cols = ["total_onshift_dashers", "total_busy_dashers", "total_outstanding_orders", "dasher_count_difference"]
op_corr = []
for col in op_cols:
    sub = df[[col, "delivery_duration_minutes"]].dropna()
    op_corr.append({
        "variavel": col, "n": len(sub),
        "pearson": round(sub[col].corr(sub["delivery_duration_minutes"], method="pearson"), 4),
        "spearman": round(sub[col].corr(sub["delivery_duration_minutes"], method="spearman"), 4),
    })
display(pd.DataFrame(op_corr))

df["busy_gt_str"] = df["busy_greater_than_onshift"].astype(str)
display(agg_stats("busy_gt_str"))

df["diff_band"] = pd.cut(df["dasher_count_difference"], bins=[-np.inf, -5, -0.01, 0.01, 5, 15, np.inf],
                         labels=["< -5", "-5 a -1", "0", "1 a 5", "6 a 15", "> 15"])
display(agg_stats("diff_band"))

## 5. Restaurantes e Mercados

In [ ]:
df["market_str"] = df["market_id"].astype("object").fillna("NA").astype(str)
display(agg_stats("market_str").sort_values("mediana", ascending=False))

cat_stats = agg_stats("store_primary_category")
cat_500 = cat_stats[cat_stats["n"] >= 500]
print(f"Categorias com n >= 500: {len(cat_500)} de {len(cat_stats)}")
display(cat_500.sort_values("mediana", ascending=False).head(10))
display(cat_500.sort_values("mediana", ascending=True).head(10))

store_stats = agg_stats("store_id")
store_100 = store_stats[store_stats["n"] >= 100]
print(f"Lojas com n >= 100: {len(store_100)} de {len(store_stats)}")
display(store_100.sort_values("mediana", ascending=False).head(10))
display(store_100.sort_values("mediana", ascending=True).head(10))

## 6. Correlações: Pearson vs Spearman

In [ ]:
num_cols = [
    "delivery_duration_minutes", "total_items", "num_distinct_items", "subtotal",
    "min_item_price", "max_item_price", "total_onshift_dashers", "total_busy_dashers",
    "total_outstanding_orders", "dasher_count_difference",
    "estimated_order_place_duration", "estimated_store_to_consumer_driving_duration"
]
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
sns.heatmap(df[num_cols].corr(method="pearson"), annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=axes[0], annot_kws={"size": 8})
axes[0].set_title("Matriz de Correlação — Pearson")
sns.heatmap(df[num_cols].corr(method="spearman"), annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=axes[1], annot_kws={"size": 8})
axes[1].set_title("Matriz de Correlação — Spearman")
fig.tight_layout()
plt.show()